# Stirling Cryocooler — Live Pipeline Demo

**Bachelor's Thesis (HVL + UPV, 2026)** — Andrés Monteagudo

This notebook runs the data-analysis pipeline on the **published Case 1**
experiment and reproduces the P-V diagram, the three pressure models and
the refrigeration performance metrics.

Tap **Runtime → Run all** (or the play button on each cell).


### 1. Get the code
Clone the repository and install the dependencies.

In [ ]:
# EDIT THIS LINE with your GitHub user/repo:
REPO = "SolarVibes-byte/Stirling-DAQ--Demo"

!git clone -q https://github.com/{REPO}.git project 2>/dev/null || (cd project && git pull -q)
!pip install -q -r project/requirements.txt
print("Code ready.")

### 2. Load the pipeline
Add the modules to the path so they import directly.

In [ ]:
import sys
sys.path.insert(0, "project/src")

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

import MathematicalProcessor as MP
print("Pipeline loaded.")

### 3. Process the published Case 1
The orchestrator reads the raw pressure curve and the slow-data log,
reconstructs the cycle, and evaluates every model on the same Fourier
volume basis.

In [ ]:
result = MP.process_case(
    "project/data/case1.txt",
    slow_path="project/data/case1_logg.txt",
)
print(f"Mode: {result.operating_mode}")
print(f"Indicated work  W_exp = {result.W_experimental_J:.1f} J   (paper: 82.6 J)")
print(f"Isothermal      W_sch = {result.W_schmidt_J:.1f} J")
print(f"Adiabatic       W_ada = {result.W_adiabatic_J:.1f} J")

### 4. P-V diagram — three pressure models
Experimental (measured) vs Isothermal and Adiabatic, all on the Sage Fourier volume.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

V = result.V_fourier_m3 * 1e6  # cm^3
order = np.argsort(result.theta_deg)
def loop(y):
    yy = y[order]; return np.append(yy, yy[0])
Vx = loop(V)

fig, ax = plt.subplots(figsize=(7, 5.2))
ax.fill(Vx, loop(result.P_experimental_bar), color="#1a1a1a", alpha=0.05)
ax.plot(Vx, loop(result.P_experimental_bar), color="#1a1a1a", lw=2.4,
        label=f"Experimental ({result.W_experimental_J:.1f} J)")
ax.plot(Vx, loop(result.P_adiabatic_bar), color="#d95f0e", lw=2.0,
        label=f"Adiabatic ({result.W_adiabatic_J:.1f} J)")
ax.plot(Vx, loop(result.P_schmidt_bar), color="#2c7fb8", lw=2.0, ls="--",
        label=f"Isothermal ({result.W_schmidt_J:.1f} J)")
ax.set_xlabel("Total volume V [cm3]  (Sage Fourier)")
ax.set_ylabel("Pressure p [bar]")
ax.set_title("P-V cycle - Sigma 1-125A, Case 1")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

### 5. Performance metrics (refrigeration mode)
Computed by a global energy balance from the measured boundary conditions.

In [ ]:
import pandas as pd
m = result.metrics
table = pd.DataFrame([
    ("Indicated work W_PV [W]",   f"{m.W_PV_W:.0f}",        "1927"),
    ("Heat rejected Q_out [W]",   f"{m.Q_out_sink_W:.0f}",  "2930"),
    ("Heat lifted Q_in [W]",      f"{m.Q_in_source_W:.0f}", "939"),
    ("COP_R (PV)",                f"{m.COP_R_PV:.3f}",      "0.473"),
    ("COP_R (system)",            f"{m.COP_R_system:.3f}",  "0.250"),
    ("COP reversible (Carnot)",   f"{m.COP_reversible:.3f}","1.262"),
    ("2nd-law eff. (PV) [%]",     f"{m.eta_II_PV*100:.1f}", "40.0"),
    ("2nd-law eff. (system) [%]", f"{m.eta_II_system*100:.1f}", "21.1"),
], columns=["Metric", "This pipeline", "Paper (Lummen 2024)"])
table

---
**Result:** the pipeline reconstructs the published cycle within ~1 % on the
indicated work, confirming it processes the same experimental data as the
reference paper. The small COP deviation reflects the ambient-leak and creep
terms not resolved by the global balance.